In [1]:
"""
M7U4 — IFC to Neo4j Graph: Extract and Load
============================================
Extracts the Duplex Apartment IFC model and loads it into Neo4j
as a property graph for data quality analysis.

Graph schema:
  - Project, Site, Building, Storey, Space (spatial hierarchy)
  - Element (with dual labels per IFC class, e.g. :Element:IfcWall)
  - PropertySet, Property
  - Type, Material
  - Relationships: CONTAINS, LOCATED_IN, HAS_PSET, HAS_PROPERTY,
                   DEFINED_BY, MADE_OF
"""

import os
from pathlib import Path
import ifcopenshell
import ifcopenshell.util.element
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Resolve paths relative to project root (one level above notebooks/)
PROJECT_ROOT = Path.cwd().parent
IFC_PATH = PROJECT_ROOT / "ifc" / "Duplex_A_20110907.ifc"

# Load Neo4j credentials from .env
load_dotenv(PROJECT_ROOT / ".env")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

print(f"IFC file:      {IFC_PATH}")
print(f"IFC exists:    {IFC_PATH.exists()}")
print(f"Neo4j URI:     {NEO4J_URI}")
print(f"Neo4j user:    {NEO4J_USERNAME}")

IFC file:      D:\Projects\m7u4-ifc-graph\ifc\Duplex_A_20110907.ifc
IFC exists:    True
Neo4j URI:     neo4j://127.0.0.1:7687
Neo4j user:    neo4j


In [2]:
ifc = ifcopenshell.open(str(IFC_PATH))

print(f"IFC schema:    {ifc.schema}")
print(f"Project name:  {ifc.by_type('IfcProject')[0].Name}")
print(f"Total entities: {len(list(ifc))}")
print()
print("Top-level entity counts:")
for ifc_class in ["IfcProject", "IfcSite", "IfcBuilding", "IfcBuildingStorey",
                   "IfcSpace", "IfcElement", "IfcWall", "IfcDoor", "IfcWindow",
                   "IfcSlab", "IfcStair", "IfcRailing", "IfcFurnishingElement",
                   "IfcCovering", "IfcOpeningElement", "IfcFlowSegment",
                   "IfcFlowTerminal", "IfcFlowFitting"]:
    count = len(ifc.by_type(ifc_class))
    if count > 0:
        print(f"  {ifc_class:30s} {count}")

IFC schema:    IFC2X3
Project name:  0001
Total entities: 38898

Top-level entity counts:
  IfcProject                     1
  IfcSite                        1
  IfcBuilding                    1
  IfcBuildingStorey              4
  IfcSpace                       21
  IfcElement                     268
  IfcWall                        57
  IfcDoor                        14
  IfcWindow                      24
  IfcSlab                        21
  IfcStair                       2
  IfcRailing                     4
  IfcFurnishingElement           61
  IfcCovering                    13
  IfcOpeningElement              50


In [3]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

def run_cypher(query, **params):
    """Run a Cypher query and return all records as a list of dicts."""
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

# Verify connection and report the current database state
result = run_cypher("MATCH (n) RETURN count(n) AS node_count")
print(f"Existing nodes in database: {result[0]['node_count']}")

Existing nodes in database: 0


In [4]:
# Clean slate every time we run the notebook end-to-end
run_cypher("MATCH (n) DETACH DELETE n")

# Drop and recreate the GlobalId uniqueness constraint
run_cypher("DROP CONSTRAINT element_globalid IF EXISTS")
run_cypher("""
    CREATE CONSTRAINT element_globalid IF NOT EXISTS
    FOR (n:Element) REQUIRE n.GlobalId IS UNIQUE
""")

print("Database cleared and constraints created.")

Database cleared and constraints created.


In [5]:
def load_spatial_structure(ifc_file):
    """Create nodes for IfcProject, IfcSite, IfcBuilding, IfcBuildingStorey, IfcSpace
    and link them with CONTAINS relationships."""
    
    spatial_classes = [
        ("IfcProject", "Project"),
        ("IfcSite", "Site"),
        ("IfcBuilding", "Building"),
        ("IfcBuildingStorey", "Storey"),
        ("IfcSpace", "Space"),
    ]
    
    created = {}
    for ifc_class, label in spatial_classes:
        for entity in ifc_file.by_type(ifc_class):
            # Use MERGE on GlobalId to avoid duplicates
            run_cypher(f"""
                MERGE (n:{label} {{GlobalId: $gid}})
                SET n.IfcClass = $ifc_class,
                    n.Name = $name,
                    n.LongName = $long_name,
                    n.Description = $description
            """,
                gid=entity.GlobalId,
                ifc_class=ifc_class,
                name=getattr(entity, "Name", None),
                long_name=getattr(entity, "LongName", None),
                description=getattr(entity, "Description", None),
            )
            created[entity.GlobalId] = label
    
    # Create CONTAINS relationships using IfcRelAggregates
    for rel in ifc_file.by_type("IfcRelAggregates"):
        parent_gid = rel.RelatingObject.GlobalId
        for child in rel.RelatedObjects:
            if child.GlobalId in created:
                run_cypher("""
                    MATCH (parent {GlobalId: $parent_gid})
                    MATCH (child {GlobalId: $child_gid})
                    MERGE (parent)-[:CONTAINS]->(child)
                """,
                    parent_gid=parent_gid,
                    child_gid=child.GlobalId,
                )
    
    return created

spatial_nodes = load_spatial_structure(ifc)
print(f"Loaded {len(spatial_nodes)} spatial nodes:")
for label in ["Project", "Site", "Building", "Storey", "Space"]:
    count = sum(1 for v in spatial_nodes.values() if v == label)
    print(f"  {label:10s} {count}")

Loaded 28 spatial nodes:
  Project    1
  Site       1
  Building   1
  Storey     4
  Space      21


In [6]:
def load_elements(ifc_file):
    """Create :Element nodes with a secondary IFC-class label,
    and link to their containing storey/space."""
    
    elements = ifc_file.by_type("IfcElement")
    
    for element in elements:
        ifc_class = element.is_a()  # e.g. "IfcWallStandardCase"
        
        # Dual label: every element is :Element AND its specific IFC class
        run_cypher(f"""
            MERGE (e:Element:{ifc_class} {{GlobalId: $gid}})
            SET e.IfcClass = $ifc_class,
                e.Name = $name,
                e.ObjectType = $object_type,
                e.Tag = $tag,
                e.Description = $description
        """,
            gid=element.GlobalId,
            ifc_class=ifc_class,
            name=getattr(element, "Name", None),
            object_type=getattr(element, "ObjectType", None),
            tag=getattr(element, "Tag", None),
            description=getattr(element, "Description", None),
        )
    
    # Spatial containment via IfcRelContainedInSpatialStructure
    for rel in ifc_file.by_type("IfcRelContainedInSpatialStructure"):
        container_gid = rel.RelatingStructure.GlobalId
        for element in rel.RelatedElements:
            run_cypher("""
                MATCH (container {GlobalId: $container_gid})
                MATCH (element:Element {GlobalId: $element_gid})
                MERGE (container)-[:CONTAINS]->(element)
            """,
                container_gid=container_gid,
                element_gid=element.GlobalId,
            )
    
    return len(elements)

n_elements = load_elements(ifc)
print(f"Loaded {n_elements} elements.")

Loaded 268 elements.


In [8]:
def load_property_sets(ifc_file):
    """Create :PropertySet nodes and :Property leaf nodes,
    linked to their host elements via HAS_PSET and HAS_PROPERTY."""
    
    n_psets = 0
    n_properties = 0
    
    for element in ifc_file.by_type("IfcObject"):
        # get_psets returns {pset_name: {prop_name: value, ...}, ...}
        psets = ifcopenshell.util.element.get_psets(element)
        
        for pset_name, properties in psets.items():
            # Skip the 'id' meta-key that ifcopenshell injects
            pset_id = properties.get("id")
            pset_gid = f"{element.GlobalId}__{pset_name}"
            
            run_cypher("""
                MERGE (ps:PropertySet {pset_id: $pset_gid})
                SET ps.Name = $pset_name
                WITH ps
                MATCH (e:Element {GlobalId: $element_gid})
                MERGE (e)-[:HAS_PSET]->(ps)
            """,
                pset_gid=pset_gid,
                pset_name=pset_name,
                element_gid=element.GlobalId,
            )
            n_psets += 1
            
            for prop_name, prop_value in properties.items():
                if prop_name == "id":
                    continue
                prop_gid = f"{pset_gid}__{prop_name}"
                
                # Coerce value to string for storage; track raw type
                value_str = "" if prop_value is None else str(prop_value)
                value_type = type(prop_value).__name__
                is_empty = prop_value is None or value_str.strip() == ""
                
                run_cypher("""
                    MERGE (p:Property {prop_id: $prop_gid})
                    SET p.Name = $prop_name,
                        p.Value = $value_str,
                        p.DataType = $value_type,
                        p.IsEmpty = $is_empty
                    WITH p
                    MATCH (ps:PropertySet {pset_id: $pset_gid})
                    MERGE (ps)-[:HAS_PROPERTY]->(p)
                """,
                    prop_gid=prop_gid,
                    prop_name=prop_name,
                    value_str=value_str,
                    value_type=value_type,
                    is_empty=is_empty,
                    pset_gid=pset_gid,
                )
                n_properties += 1
    
    return n_psets, n_properties

n_psets, n_props = load_property_sets(ifc)
print(f"Loaded {n_psets} property set instances and {n_props} properties.")

Loaded 2388 property set instances and 13455 properties.


In [9]:
def load_types_and_materials(ifc_file):
    """Create :Type nodes (IfcTypeObject) and :Material nodes,
    and link elements to them."""
    
    # Type relationships
    n_types = 0
    for rel in ifc_file.by_type("IfcRelDefinesByType"):
        type_obj = rel.RelatingType
        run_cypher("""
            MERGE (t:Type {GlobalId: $gid})
            SET t.IfcClass = $ifc_class,
                t.Name = $name
        """,
            gid=type_obj.GlobalId,
            ifc_class=type_obj.is_a(),
            name=getattr(type_obj, "Name", None),
        )
        n_types += 1
        for related in rel.RelatedObjects:
            run_cypher("""
                MATCH (e:Element {GlobalId: $element_gid})
                MATCH (t:Type {GlobalId: $type_gid})
                MERGE (e)-[:DEFINED_BY]->(t)
            """,
                element_gid=related.GlobalId,
                type_gid=type_obj.GlobalId,
            )
    
    # Material relationships
    n_materials = 0
    for rel in ifc_file.by_type("IfcRelAssociatesMaterial"):
        material = rel.RelatingMaterial
        # Material can be IfcMaterial, IfcMaterialLayerSet, etc.
        # We pull the simple name if available
        if material.is_a("IfcMaterial"):
            mat_name = material.Name
        elif hasattr(material, "Name"):
            mat_name = material.Name or material.is_a()
        else:
            mat_name = material.is_a()
        
        run_cypher("""
            MERGE (m:Material {Name: $name})
        """, name=mat_name)
        n_materials += 1
        
        for related in rel.RelatedObjects:
            if related.is_a("IfcElement"):
                run_cypher("""
                    MATCH (e:Element {GlobalId: $element_gid})
                    MATCH (m:Material {Name: $name})
                    MERGE (e)-[:MADE_OF]->(m)
                """,
                    element_gid=related.GlobalId,
                    name=mat_name,
                )
    
    return n_types, n_materials

n_types, n_materials = load_types_and_materials(ifc)
print(f"Loaded {n_types} type associations and {n_materials} material associations.")

Loaded 37 type associations and 92 material associations.


In [10]:
print("=" * 50)
print("Graph load complete. Summary:")
print("=" * 50)

# Count nodes by label
results = run_cypher("""
    MATCH (n)
    UNWIND labels(n) AS label
    RETURN label, count(*) AS count
    ORDER BY count DESC
""")
print("\nNode counts by label:")
for r in results:
    print(f"  {r['label']:30s} {r['count']}")

# Count relationships by type
results = run_cypher("""
    MATCH ()-[r]->()
    RETURN type(r) AS rel_type, count(*) AS count
    ORDER BY count DESC
""")
print("\nRelationship counts by type:")
for r in results:
    print(f"  {r['rel_type']:30s} {r['count']}")

# Close the driver
driver.close()
print("\nDriver closed. Load complete.")

Graph load complete. Summary:

Node counts by label:
  Property                       13455
  PropertySet                    2388
  Element                        268
  IfcFurnishingElement           61
  IfcWallStandardCase            56
  IfcOpeningElement              50
  Type                           37
  IfcWindow                      24
  Space                          21
  IfcSlab                        21
  IfcDoor                        14
  IfcCovering                    13
  IfcBeam                        8
  IfcFooting                     7
  Storey                         4
  IfcMember                      4
  IfcRailing                     4
  IfcStair                       2
  IfcStairFlight                 2
  Material                       2
  Project                        1
  Site                           1
  Building                       1
  IfcRoof                        1
  IfcWall                        1

Relationship counts by type:
  HAS_PROPERTY          